## Movimiento Run and Tumble

## Teoría: Movimiento Run and Tumble

El movimiento 'Run and Tumble' es un modelo estocástico utilizado para describir el movimiento de bacterias como *E. coli*. En este modelo, la partícula (o bacteria) alterna entre dos comportamientos:

- **Run (correr):** Se mueve en línea recta a velocidad constante $v$ durante un tiempo aleatorio.
- **Tumble (giro):** Cambia aleatoriamente de dirección (en 1D, invierte el sentido).

El proceso se caracteriza por una probabilidad $p_{tumble}$ de cambiar de dirección en cada paso de tiempo $\Delta t$. El resultado es una trayectoria que alterna segmentos rectos y cambios bruscos de dirección.

Matemáticamente, el modelo puede describirse como una caminata aleatoria persistente. El desplazamiento medio cuadrático (MSD) para este proceso en 1D está dado por:

$$
\langle x^2(t) \rangle = \frac{v^2}{\lambda^2} \left( \lambda t - 1 + e^{-2\lambda t} \right)
$$

donde $\lambda = p_{tumble}/\Delta t$ es la tasa de cambio de dirección.

Este modelo es fundamental para entender la motilidad bacteriana y la transición entre movimiento balístico y difusivo.

In [ ]:
using Random, Distributions, Statistics, Plots #Paqueterías a usar

In [ ]:
N_steps, Δt, v, x0, p_tumble, num_trajectories = Int64(5e3), 0.1, 1.0, 0.0, 0.1, Int64(1e3) #Parámetros de la simulación

In [ ]:
function run_and_tumble_generator(N_steps, Δt, v, x0, p_tumble) #Función para generar una trayectoria del modelo run-and-tumble
    trajectory = zeros(N_steps)
    trajectory[1] = x0
    s = rand((1, -1))
    for i in 2:N_steps
        trajectory[i] = trajectory[i-1] + v*s* Δt
        if rand() < p_tumble
            s *= -1
        end
    end
    return trajectory
end

In [ ]:
function run_and_tumble_trajectories(N_steps, Δt, v, x0, p_tumble, num_trajectories) #Función para generar múltiples trayectorias del modelo run-and-tumble
    trajectories = []
    for _ in 1:num_trajectories
        push!(trajectories, run_and_tumble_generator(N_steps, Δt, v, x0, p_tumble))
    end
    return trajectories
end

In [ ]:
function plot_trajectories(trajectories) #Función para graficar las trayectorias generadas
    plot()
    for traj in trajectories
        plot!(traj, lw=1, alpha=0.3, label=false)
    end
    xlabel!("Time Steps")
    ylabel!("Position")
    title!("Run and Tumble Trajectories")
    display(current())
end

In [ ]:
function compute_msd(trajectories) #Función para calcular el MSD a partir de múltiples trayectorias
    msd_sum = zeros(N_steps)
    
    for i in 1:num_trajectories
        traj = trajectories[i]
        msd_sum .+= traj.^2
    end
    msd = msd_sum / num_trajectories
    return msd
end

In [ ]:
trajectories = run_and_tumble_trajectories(N_steps, Δt, v, x0, p_tumble, num_trajectories)
plot_trajectories(trajectories)

In [ ]:
function theoretical_msd(t, v, λ)
    return v^2 * (λ*t - (1 - exp(-2 * λ * t)) )/ (λ^2)
end

In [ ]:
function plot_msd(msd, N_steps, Δt, v, p_tumble)
    plot(msd, lw=2, label="MSD")
    plot!(theoretical_msd.((0:N_steps-1)*Δt, v, p_tumble/Δt), lw=2, ls=:dash, label="Theoretical MSD", color=:red)

    xlabel!("Time Steps")
    ylabel!("Mean Squared Displacement")
    title!("Mean Squared Displacement of Run and Tumble")
    display(current())
end

In [ ]:
plot_msd(compute_msd(trajectories), N_steps, Δt, v, p_tumble)